# 03-1 · The satisfaction matrix S

| | |
|---|---|
| **What it does** | scores every member against every option in F: `s_i(x) = cosine(taste_i, taste_x)` |
| **Proves** | the input every selector in 03-2 reduces to one number |
| **Eval surface** | the range of S — watch how compressed it is |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## Rebuild the group and F from 02-1


In [3]:
# Step 1: the same 4-member group and feasible set as 02-safety/01
personas = json.load(open(nbio.SMALL_PERSONAS))
group_names = ["Ravi", "Marco", "Omar", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"],
                     "taste": p["taste_vector"]})

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip()
            return
        for v in node.values():
            yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node:
            yield from iter_dish_names(v)

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        name = row.get("ingredient", "").strip().lower()
        if name:
            flavor_map[name] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def infer_taste(dish_name):
    matched = [flavor_map[ing] for ing in flavor_map if ing in dish_name.lower()]
    return [sum(m[d] for m in matched) / len(matched) for d in nbio.TASTE_DIMS] if matched else [0.0]*6

catalog = []
for f in sorted(nbio.SMALL_MENUS.glob("*.json")):
    payload = json.loads(f.read_text())
    names = [n for n in iter_dish_names(payload) if n]
    vecs = [infer_taste(n) for n in names]
    nz = [v for v in vecs if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6
    catalog.append({"option_id": f.stem, "name": f.stem,
                     "menu": [{"name": n} for n in names], "taste": avg})

F = nbio.build_feasible_set(catalog, members)
print(f"members: {[m['display_name'] for m in members]}  |  F size: {len(F)}")


members: ['Ravi', 'Marco', 'Omar', 'Alice']  |  F size: 5


## Score every member against every option

`nbio.taste_similarity(member_taste, option_taste)` per member, per option. The
grid this builds is S.


In [4]:
# Step 2: build S
member_ids = [m["member_id"] for m in members]
names = {m["member_id"]: m["display_name"] for m in members}
labels = {o["option_id"]: o["name"] for o in F}

S = {}
for o in F:
    S[o["option_id"]] = {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"]) for m in members}

before = {"F": len(F)}
nbio.io(2, "satisfaction_matrix", before, {"F": len(F), "S": len(S)},
        reads=["F (option taste)", "members (taste)"])
nbio.matrix(S, member_ids, names=names, labels=labels)

vals = [v for row in S.values() for v in row.values()]
print(f"\nrange {min(vals):.3f} .. {max(vals):.3f}   spread {max(vals)-min(vals):.3f}")


+- STEP 2  satisfaction_matrix                                      [py]
| IN      : F (option taste), members (taste)
| NEW     : S
| CHANGED : -
+-------------------------------------------------------------------------
option                              Ravi     Marco      Omar     Alice      min      mean
-----------------------------------------------------------------------------------------
Arabian Corner Restaurant -...     0.501     0.633     0.709     0.787    0.501    0.658
Casa.cafe                          0.624     0.746     0.825     0.849    0.624    0.761
Delicia Multi Cuisine Resta...     0.622     0.747     0.834     0.833    0.622    0.759
House of Chefs                     0.540     0.704     0.774     0.801    0.540    0.705
Ministry of South Bar and K...     0.529     0.670     0.755     0.801    0.529    0.689

range 0.501 .. 0.849   spread 0.348


### Reading it

Cosine similarity over non-negative 6-D vectors cannot go low — with no negative
components every score lands in roughly 0.5-1.0. "Satisfaction" therefore has far
less dynamic range than a 0-1 scale suggests; **differences between options
matter more than levels**. Carry `S`, `members`, and `F` forward — **03-2** turns
this grid into a single pick.


## Revision: is C1 robust to the choice of satisfaction function?

Everything above uses one satisfaction function, the cosine between a diner's
taste and the restaurant's whole-menu taste. This section repeats the C1
analysis on the Food.com diners and the Uber Eats Houston and Seattle catalogs
(200 seeded groups each) under five alternatives: cosine to the diner's own safe
dishes, Euclidean similarity, centred cosine, and two order-aware utilities (the
diner's best safe dish, and the mean of their top-3 safe dishes).

For each utility it sweeps lambda over a fine grid and reports the share of the
achievable worst-off gain that Fair recovers (achievable = Least-Misery's MinSat
minus Average's), at the largest lambda whose cost stays within a third of
Least-Misery's cost, and at lambda 5. It then evaluates picks made on the paper's
proxy under the other utilities (E1b), and checks how the result depends on the
candidate-set size (E2).

Reads the per-group matrices (`nbio.rev_data()`, built on first use; see
`06-evaluate/01`). Writes `figdata_rev/e1_utilities.csv`, `e1_frontier.csv`,
`e1b_cross_eval.csv` and `e2_candidate_size.csv`.

In [5]:
import os, csv
import numpy as np
from nbio import REV_CITIES, REV_LAMBDAS, rev_data, rev_c1_summary, rev_pick, rev_boot_ci

OUT = nbio.REV_OUT
os.makedirs(OUT, exist_ok=True)
D = rev_data()
UTILS = [("S", "cosine, whole menu (paper)"), ("Dsafe", "cosine, safe-subset menu"),
         ("Se", "Euclidean similarity"), ("Sc", "centred cosine"),
         ("D3", "order-aware, mean of top-3 safe dishes"), ("D", "order-aware, best safe dish")]
GRID = sorted(set(list(np.round(np.logspace(-2, 3, 61), 4)) + REV_LAMBDAS))


def frontier(Ms):
    """% achievable vs cost share of LM, over the lambda grid."""
    rows = []
    for lam in GRID:
        r = rev_c1_summary(Ms, lam)
        rows.append((lam, r["pct"], 100 * r["cost"] / r["cost_lm"] if r["cost_lm"] else np.nan, r))
    return rows


def at_cost_share(rows, target=100 / 3):
    """Largest lambda whose cost share does not exceed target."""
    ok = [r for r in rows if not np.isnan(r[2]) and r[2] <= target]
    return max(ok, key=lambda r: r[0])


def write(name, rows):
    with open(f"{OUT}/{name}", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

In [6]:
# E1: the C1 frontier under each satisfaction function
out1, out_front = [], []
for city in REV_CITIES:
    for key, label in UTILS:
        Ms = [g[key] for g in D[city]]
        fr = frontier(Ms)
        for lam, pct, cs, r in fr:
            out_front.append(dict(city=city, utility=key, lam=lam, pct=round(pct, 2), cost_share=round(cs, 2)))
        lam_m, pct_m, cs_m, rm = at_cost_share(fr)
        r5 = rev_c1_summary(Ms, 5)
        # selector ordering on MinSat / Jain
        order = {}
        for rule in ["average", "least_misery", "most_pleasure", "fair"]:
            ps = [rev_pick(M, rule, 0.5 if key != "Sc" else 0.5) for M in Ms]
            order[rule] = (np.mean([M[p].min() for M, p in zip(Ms, ps)]),
                           np.mean([M[p].mean() for M, p in zip(Ms, ps)]))
        out1.append(dict(city=city, utility=key, label=label,
                         achievable=round(rm["achievable"], 4),
                         lam_matched=lam_m, pct_matched=round(pct_m, 1),
                         pct_matched_lo=round(rm["pct_lo"], 1), pct_matched_hi=round(rm["pct_hi"], 1),
                         cost_share_matched=round(cs_m, 1),
                         gain_matched=round(rm["gain"], 4),
                         pct_lam5=round(r5["pct"], 1), cost_share_lam5=round(100 * r5["cost"] / r5["cost_lm"], 1),
                         mp_minsat=round(order["most_pleasure"][0], 4), avg_minsat=round(order["average"][0], 4),
                         lm_minsat=round(order["least_misery"][0], 4)))
        print(out1[-1])

write("e1_utilities.csv", out1)
write("e1_frontier.csv", out_front)

{'city': 'Houston, TX', 'utility': 'S', 'label': 'cosine, whole menu (paper)', 'achievable': np.float64(0.0402), 'lam_matched': np.float64(3.8312), 'pct_matched': np.float64(58.0), 'pct_matched_lo': 51.3, 'pct_matched_hi': 64.1, 'cost_share_matched': np.float64(27.4), 'gain_matched': np.float64(0.0233), 'pct_lam5': np.float64(67.1), 'cost_share_lam5': np.float64(36.4), 'mp_minsat': np.float64(0.8103), 'avg_minsat': np.float64(0.8857), 'lm_minsat': np.float64(0.9258)}


{'city': 'Houston, TX', 'utility': 'Dsafe', 'label': 'cosine, safe-subset menu', 'achievable': np.float64(0.0384), 'lam_matched': np.float64(4.6416), 'pct_matched': np.float64(65.1), 'pct_matched_lo': 59.0, 'pct_matched_hi': 70.3, 'cost_share_matched': np.float64(33.1), 'gain_matched': np.float64(0.025), 'pct_lam5': np.float64(67.3), 'cost_share_lam5': np.float64(35.2), 'mp_minsat': np.float64(0.8006), 'avg_minsat': np.float64(0.8878), 'lm_minsat': np.float64(0.9261)}


{'city': 'Houston, TX', 'utility': 'Se', 'label': 'Euclidean similarity', 'achievable': np.float64(0.0166), 'lam_matched': np.float64(12.1153), 'pct_matched': np.float64(46.8), 'pct_matched_lo': 39.8, 'pct_matched_hi': 53.6, 'cost_share_matched': np.float64(28.5), 'gain_matched': np.float64(0.0078), 'pct_lam5': np.float64(18.9), 'cost_share_lam5': np.float64(3.0), 'mp_minsat': np.float64(0.8843), 'avg_minsat': np.float64(0.9116), 'lm_minsat': np.float64(0.9282)}


{'city': 'Houston, TX', 'utility': 'Sc', 'label': 'centred cosine', 'achievable': np.float64(0.3708), 'lam_matched': np.float64(0.8254), 'pct_matched': np.float64(66.3), 'pct_matched_lo': 62.3, 'pct_matched_hi': 70.3, 'cost_share_matched': np.float64(29.4), 'gain_matched': np.float64(0.2457), 'pct_lam5': np.float64(92.0), 'cost_share_lam5': np.float64(105.2), 'mp_minsat': np.float64(-0.5004), 'avg_minsat': np.float64(-0.1577), 'lm_minsat': np.float64(0.2131)}


{'city': 'Houston, TX', 'utility': 'D3', 'label': 'order-aware, mean of top-3 safe dishes', 'achievable': np.float64(0.0034), 'lam_matched': 50, 'pct_matched': np.float64(71.3), 'pct_matched_lo': 59.5, 'pct_matched_hi': 80.5, 'cost_share_matched': np.float64(30.4), 'gain_matched': np.float64(0.0024), 'pct_lam5': np.float64(32.6), 'cost_share_lam5': np.float64(5.8), 'mp_minsat': np.float64(0.9429), 'avg_minsat': np.float64(0.9772), 'lm_minsat': np.float64(0.9806)}


{'city': 'Houston, TX', 'utility': 'D', 'label': 'order-aware, best safe dish', 'achievable': np.float64(0.0016), 'lam_matched': np.float64(68.1292), 'pct_matched': np.float64(59.6), 'pct_matched_lo': 47.8, 'pct_matched_hi': 70.3, 'cost_share_matched': np.float64(25.9), 'gain_matched': np.float64(0.001), 'pct_lam5': np.float64(3.6), 'cost_share_lam5': np.float64(0.1), 'mp_minsat': np.float64(0.9556), 'avg_minsat': np.float64(0.9855), 'lm_minsat': np.float64(0.9871)}


... (2 outputs left out)

{'city': 'Seattle, WA', 'utility': 'Se', 'label': 'Euclidean similarity', 'achievable': np.float64(0.0118), 'lam_matched': np.float64(14.678), 'pct_matched': np.float64(48.4), 'pct_matched_lo': 41.3, 'pct_matched_hi': 55.1, 'cost_share_matched': np.float64(25.2), 'gain_matched': np.float64(0.0057), 'pct_lam5': np.float64(16.0), 'cost_share_lam5': np.float64(3.9), 'mp_minsat': np.float64(0.8841), 'avg_minsat': np.float64(0.9169), 'lm_minsat': np.float64(0.9287)}


{'city': 'Seattle, WA', 'utility': 'Sc', 'label': 'centred cosine', 'achievable': np.float64(0.4028), 'lam_matched': np.float64(0.8254), 'pct_matched': np.float64(68.7), 'pct_matched_lo': 64.4, 'pct_matched_hi': 72.9, 'cost_share_matched': np.float64(31.8), 'gain_matched': np.float64(0.2769), 'pct_lam5': np.float64(93.5), 'cost_share_lam5': np.float64(101.5), 'mp_minsat': np.float64(-0.4953), 'avg_minsat': np.float64(-0.1859), 'lm_minsat': np.float64(0.2169)}


{'city': 'Seattle, WA', 'utility': 'D3', 'label': 'order-aware, mean of top-3 safe dishes', 'achievable': np.float64(0.0018), 'lam_matched': np.float64(68.1292), 'pct_matched': np.float64(57.3), 'pct_matched_lo': 45.7, 'pct_matched_hi': 67.1, 'cost_share_matched': np.float64(29.8), 'gain_matched': np.float64(0.001), 'pct_lam5': np.float64(7.0), 'cost_share_lam5': np.float64(0.2), 'mp_minsat': np.float64(0.9457), 'avg_minsat': np.float64(0.9777), 'lm_minsat': np.float64(0.9794)}


{'city': 'Seattle, WA', 'utility': 'D', 'label': 'order-aware, best safe dish', 'achievable': np.float64(0.0017), 'lam_matched': np.float64(82.5404), 'pct_matched': np.float64(73.0), 'pct_matched_lo': 62.1, 'pct_matched_hi': 81.7, 'cost_share_matched': np.float64(32.1), 'gain_matched': np.float64(0.0012), 'pct_lam5': np.float64(3.3), 'cost_share_lam5': np.float64(0.2), 'mp_minsat': np.float64(0.9593), 'avg_minsat': np.float64(0.9847), 'lm_minsat': np.float64(0.9864)}


In [7]:
# E1b: picks selected on the paper's proxy S, evaluated under other utilities
out2 = []
for city in REV_CITIES:
    for key, label in UTILS[1:]:
        Ms = [g["S"] for g in D[city]]; Es = [g[key] for g in D[city]]
        for lam in (0.5, 5):
            gains, lmg, mf = [], [], []
            for M, E in zip(Ms, Es):
                a = rev_pick(M, "average"); f = rev_pick(M, "fair", lam); l = rev_pick(M, "least_misery")
                gains.append(E[f].min() - E[a].min()); lmg.append(E[l].min() - E[a].min())
                mf.append(E[f].mean() - E[a].mean())
            lo, hi = rev_boot_ci(gains)
            # oracle achievable under E
            orc = np.mean([E.min(1).max() - E[rev_pick(M, "average")].min() for M, E in zip(Ms, Es)])
            out2.append(dict(city=city, eval_utility=key, lam=lam, gain=round(np.mean(gains), 4),
                             lo=round(lo, 4), hi=round(hi, 4), lm_gain=round(np.mean(lmg), 4),
                             oracle_gap=round(orc, 4),
                             pct_of_oracle=round(100 * np.sum(gains) / (orc * len(gains)), 1),
                             mean_change=round(np.mean(mf), 4)))
            print(out2[-1])
write("e1b_cross_eval.csv", out2)

{'city': 'Houston, TX', 'eval_utility': 'Dsafe', 'lam': 0.5, 'gain': np.float64(0.0044), 'lo': 0.0014, 'hi': 0.0077, 'lm_gain': np.float64(0.0346), 'oracle_gap': np.float64(0.0545), 'pct_of_oracle': np.float64(8.1), 'mean_change': np.float64(-0.0003)}
{'city': 'Houston, TX', 'eval_utility': 'Dsafe', 'lam': 5, 'gain': np.float64(0.027), 'lo': 0.0197, 'hi': 0.0345, 'lm_gain': np.float64(0.0346), 'oracle_gap': np.float64(0.0545), 'pct_of_oracle': np.float64(49.5), 'mean_change': np.float64(-0.0032)}
{'city': 'Houston, TX', 'eval_utility': 'Se', 'lam': 0.5, 'gain': np.float64(0.0013), 'lo': 0.0005, 'hi': 0.0022, 'lm_gain': np.float64(0.0042), 'oracle_gap': np.float64(0.014), 'pct_of_oracle': np.float64(9.6), 'mean_change': np.float64(-0.0005)}


{'city': 'Houston, TX', 'eval_utility': 'Se', 'lam': 5, 'gain': np.float64(0.0055), 'lo': 0.004, 'hi': 0.0071, 'lm_gain': np.float64(0.0042), 'oracle_gap': np.float64(0.014), 'pct_of_oracle': np.float64(39.5), 'mean_change': np.float64(-0.0046)}


{'city': 'Houston, TX', 'eval_utility': 'Sc', 'lam': 0.5, 'gain': np.float64(0.0041), 'lo': -0.0031, 'hi': 0.0127, 'lm_gain': np.float64(0.0264), 'oracle_gap': np.float64(0.5338), 'pct_of_oracle': np.float64(0.8), 'mean_change': np.float64(-0.0008)}
{'city': 'Houston, TX', 'eval_utility': 'Sc', 'lam': 5, 'gain': np.float64(0.0174), 'lo': -0.003, 'hi': 0.0371, 'lm_gain': np.float64(0.0264), 'oracle_gap': np.float64(0.5338), 'pct_of_oracle': np.float64(3.3), 'mean_change': np.float64(-0.0186)}
{'city': 'Houston, TX', 'eval_utility': 'D3', 'lam': 0.5, 'gain': np.float64(-0.0033), 'lo': -0.0077, 'hi': -0.0001, 'lm_gain': np.float64(-0.0068), 'oracle_gap': np.float64(0.0504), 'pct_of_oracle': np.float64(-6.5), 'mean_change': np.float64(-0.0008)}


{'city': 'Houston, TX', 'eval_utility': 'D3', 'lam': 5, 'gain': np.float64(-0.0048), 'lo': -0.012, 'hi': 0.0023, 'lm_gain': np.float64(-0.0068), 'oracle_gap': np.float64(0.0504), 'pct_of_oracle': np.float64(-9.5), 'mean_change': np.float64(-0.0002)}


{'city': 'Houston, TX', 'eval_utility': 'D', 'lam': 0.5, 'gain': np.float64(-0.0015), 'lo': -0.0034, 'hi': -0.0002, 'lm_gain': np.float64(-0.0048), 'oracle_gap': np.float64(0.0379), 'pct_of_oracle': np.float64(-4.1), 'mean_change': np.float64(-0.0004)}
{'city': 'Houston, TX', 'eval_utility': 'D', 'lam': 5, 'gain': np.float64(-0.0015), 'lo': -0.0066, 'hi': 0.0039, 'lm_gain': np.float64(-0.0048), 'oracle_gap': np.float64(0.0379), 'pct_of_oracle': np.float64(-4.0), 'mean_change': np.float64(0.0004)}
{'city': 'Seattle, WA', 'eval_utility': 'Dsafe', 'lam': 0.5, 'gain': np.float64(0.0018), 'lo': -0.0018, 'hi': 0.005, 'lm_gain': np.float64(0.0348), 'oracle_gap': np.float64(0.0504), 'pct_of_oracle': np.float64(3.5), 'mean_change': np.float64(-0.0003)}


{'city': 'Seattle, WA', 'eval_utility': 'Dsafe', 'lam': 5, 'gain': np.float64(0.0207), 'lo': 0.0143, 'hi': 0.0272, 'lm_gain': np.float64(0.0348), 'oracle_gap': np.float64(0.0504), 'pct_of_oracle': np.float64(41.2), 'mean_change': np.float64(-0.0032)}


{'city': 'Seattle, WA', 'eval_utility': 'Se', 'lam': 0.5, 'gain': np.float64(0.0006), 'lo': 0.0003, 'hi': 0.0011, 'lm_gain': np.float64(-0.0019), 'oracle_gap': np.float64(0.0102), 'pct_of_oracle': np.float64(6.1), 'mean_change': np.float64(-0.0003)}
{'city': 'Seattle, WA', 'eval_utility': 'Se', 'lam': 5, 'gain': np.float64(0.0024), 'lo': 0.0014, 'hi': 0.0034, 'lm_gain': np.float64(-0.0019), 'oracle_gap': np.float64(0.0102), 'pct_of_oracle': np.float64(23.3), 'mean_change': np.float64(-0.003)}
{'city': 'Seattle, WA', 'eval_utility': 'Sc', 'lam': 0.5, 'gain': np.float64(0.0031), 'lo': -0.0091, 'hi': 0.0141, 'lm_gain': np.float64(-0.0005), 'oracle_gap': np.float64(0.5165), 'pct_of_oracle': np.float64(0.6), 'mean_change': np.float64(-0.0053)}


{'city': 'Seattle, WA', 'eval_utility': 'Sc', 'lam': 5, 'gain': np.float64(0.0207), 'lo': 0.0004, 'hi': 0.0403, 'lm_gain': np.float64(-0.0005), 'oracle_gap': np.float64(0.5165), 'pct_of_oracle': np.float64(4.0), 'mean_change': np.float64(-0.0189)}


{'city': 'Seattle, WA', 'eval_utility': 'D3', 'lam': 0.5, 'gain': np.float64(-0.0026), 'lo': -0.008, 'hi': 0.0011, 'lm_gain': np.float64(-0.0011), 'oracle_gap': np.float64(0.0554), 'pct_of_oracle': np.float64(-4.8), 'mean_change': np.float64(-0.0011)}
{'city': 'Seattle, WA', 'eval_utility': 'D3', 'lam': 5, 'gain': np.float64(0.0004), 'lo': -0.0069, 'hi': 0.0072, 'lm_gain': np.float64(-0.0011), 'oracle_gap': np.float64(0.0554), 'pct_of_oracle': np.float64(0.6), 'mean_change': np.float64(-0.0012)}
{'city': 'Seattle, WA', 'eval_utility': 'D', 'lam': 0.5, 'gain': np.float64(-0.0012), 'lo': -0.0039, 'hi': 0.0013, 'lm_gain': np.float64(-0.0009), 'oracle_gap': np.float64(0.0414), 'pct_of_oracle': np.float64(-2.9), 'mean_change': np.float64(-0.0005)}


{'city': 'Seattle, WA', 'eval_utility': 'D', 'lam': 5, 'gain': np.float64(-0.0013), 'lo': -0.0066, 'hi': 0.0042, 'lm_gain': np.float64(-0.0009), 'oracle_gap': np.float64(0.0414), 'pct_of_oracle': np.float64(-3.1), 'mean_change': np.float64(-0.0013)}


In [8]:
# E2: candidate-set size, whole-menu vs order-aware utility
out3 = []
rng = np.random.default_rng(7)
for city in REV_CITIES:
    for K in (10, 30, 100, None):
        for key in ("S", "D3", "D"):
            Ms = []
            for g in D[city]:
                M = g[key]
                if K is not None:
                    idx = np.sort(np.random.default_rng(int(g["group_id"][1:]) + 7).choice(len(M), K, replace=False))
                    M = M[idx]
                Ms.append(M)
            r = rev_c1_summary(Ms, 5)
            out3.append(dict(city=city, K=K or "all", utility=key, achievable=round(r["achievable"], 4),
                             gain_lam5=round(r["gain"], 4), lo=round(r["gain_lo"], 4), hi=round(r["gain_hi"], 4),
                             pct_lam5=round(r["pct"], 1), cost_lam5=round(r["cost"], 4), cost_lm=round(r["cost_lm"], 4)))
            print(out3[-1])
write("e2_candidate_size.csv", out3)

{'city': 'Houston, TX', 'K': 10, 'utility': 'S', 'achievable': np.float64(0.0102), 'gain_lam5': np.float64(0.0044), 'lo': 0.0016, 'hi': 0.008, 'pct_lam5': np.float64(43.5), 'cost_lam5': np.float64(-0.0008), 'cost_lm': np.float64(-0.0039)}
{'city': 'Houston, TX', 'K': 10, 'utility': 'D3', 'achievable': np.float64(0.0019), 'gain_lam5': np.float64(0.0003), 'lo': 0.0001, 'hi': 0.0007, 'pct_lam5': np.float64(16.5), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0012)}


{'city': 'Houston, TX', 'K': 10, 'utility': 'D', 'achievable': np.float64(0.001), 'gain_lam5': np.float64(0.0003), 'lo': 0.0001, 'hi': 0.0006, 'pct_lam5': np.float64(32.8), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0007)}


{'city': 'Houston, TX', 'K': 30, 'utility': 'S', 'achievable': np.float64(0.0177), 'gain_lam5': np.float64(0.0094), 'lo': 0.0055, 'hi': 0.0138, 'pct_lam5': np.float64(53.1), 'cost_lam5': np.float64(-0.0014), 'cost_lm': np.float64(-0.0073)}
{'city': 'Houston, TX', 'K': 30, 'utility': 'D3', 'achievable': np.float64(0.0024), 'gain_lam5': np.float64(0.0004), 'lo': 0.0001, 'hi': 0.0009, 'pct_lam5': np.float64(17.7), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0015)}


{'city': 'Houston, TX', 'K': 30, 'utility': 'D', 'achievable': np.float64(0.0017), 'gain_lam5': np.float64(0.0002), 'lo': 0.0, 'hi': 0.0005, 'pct_lam5': np.float64(14.5), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0008)}


{'city': 'Houston, TX', 'K': 100, 'utility': 'S', 'achievable': np.float64(0.0342), 'gain_lam5': np.float64(0.0238), 'lo': 0.0174, 'hi': 0.0305, 'pct_lam5': np.float64(69.5), 'cost_lam5': np.float64(-0.003), 'cost_lm': np.float64(-0.0092)}
{'city': 'Houston, TX', 'K': 100, 'utility': 'D3', 'achievable': np.float64(0.003), 'gain_lam5': np.float64(0.0007), 'lo': 0.0001, 'hi': 0.0016, 'pct_lam5': np.float64(24.0), 'cost_lam5': np.float64(-0.0001), 'cost_lm': np.float64(-0.0013)}


{'city': 'Houston, TX', 'K': 100, 'utility': 'D', 'achievable': np.float64(0.0019), 'gain_lam5': np.float64(0.0004), 'lo': 0.0, 'hi': 0.0009, 'pct_lam5': np.float64(21.6), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.001)}


... (6 outputs left out)

{'city': 'Seattle, WA', 'K': 100, 'utility': 'S', 'achievable': np.float64(0.029), 'gain_lam5': np.float64(0.0185), 'lo': 0.0136, 'hi': 0.0237, 'pct_lam5': np.float64(63.8), 'cost_lam5': np.float64(-0.0021), 'cost_lm': np.float64(-0.0084)}
{'city': 'Seattle, WA', 'K': 100, 'utility': 'D3', 'achievable': np.float64(0.0022), 'gain_lam5': np.float64(0.0001), 'lo': 0.0, 'hi': 0.0002, 'pct_lam5': np.float64(3.9), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0015)}


{'city': 'Seattle, WA', 'K': 100, 'utility': 'D', 'achievable': np.float64(0.0013), 'gain_lam5': np.float64(0.0001), 'lo': 0.0, 'hi': 0.0002, 'pct_lam5': np.float64(6.3), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0008)}


{'city': 'Seattle, WA', 'K': 'all', 'utility': 'S', 'achievable': np.float64(0.0358), 'gain_lam5': np.float64(0.0222), 'lo': 0.0171, 'hi': 0.0276, 'pct_lam5': np.float64(61.9), 'cost_lam5': np.float64(-0.0028), 'cost_lm': np.float64(-0.0094)}
{'city': 'Seattle, WA', 'K': 'all', 'utility': 'D3', 'achievable': np.float64(0.0018), 'gain_lam5': np.float64(0.0001), 'lo': 0.0, 'hi': 0.0003, 'pct_lam5': np.float64(7.0), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0015)}


{'city': 'Seattle, WA', 'K': 'all', 'utility': 'D', 'achievable': np.float64(0.0017), 'gain_lam5': np.float64(0.0001), 'lo': 0.0, 'hi': 0.0001, 'pct_lam5': np.float64(3.3), 'cost_lam5': np.float64(-0.0), 'cost_lm': np.float64(-0.0008)}


## Revision: a multi-criteria utility from real catalog attributes

The utility is replaced by a weighted mix of three criteria built from the real
catalog: taste (the whole-menu cosine), price (the share of the diner's safe
priced dishes within a per-diner budget) and choice (how many safe dishes the
diner has, log-scaled). Budgets and criterion weights are synthetic and seeded.
The aggregation rules and the mediator are unchanged; only the utility matrix
differs. The section reports the same C1 frontier as above and the mediator's
agreement rate with graded reactions, the LLM-parser replay and memory, with
exact and noisy (sd 0.02) reactions.

Writes `figdata_rev/e7_multicriteria.csv` and `e7_frontier.csv`. The last cell
turns `e1_utilities.csv` and `e7_multicriteria.csv` into the LaTeX table rows
`figdata_rev/tex/tab_utilities.tex`, and `e1_frontier.csv` / `e7_frontier.csv`
into the per-utility frontier CSVs `figdata_rev/tex/front_{hou,sea}_*.csv` that
the frontier figure plots.

In [9]:
from nbio import REV_MARGIN, rev_mediate, rev_graded, rev_parser_replay

GRID = sorted(set(list(np.round(np.logspace(-2, 3, 61), 4)) + REV_LAMBDAS))


def mc_utility(g, rng, med_price):
    n = g["n"]
    w = rng.dirichlet(8 * np.array([0.6, 0.25, 0.15]), size=n)          # taste, price, choice
    budget = np.exp(rng.normal(np.log(med_price), 0.35, size=n))
    nF = len(g["ids"])
    A = np.full((nF, n), 0.5); C = np.zeros((nF, n))
    for a in range(nF):
        for i in range(n):
            pr = g["prices"][a][i]
            if len(pr):
                A[a, i] = float(np.mean(pr <= budget[i]))
            C[a, i] = min(1.0, np.log1p(g["nsafe"][a, i]) / np.log1p(30))
    U = w[:, 0] * g["S"] + w[:, 1] * A + w[:, 2] * C
    return U, A, C

In [10]:
rows, front = [], []
for city in REV_CITIES:
    allp = np.concatenate([p for g in D[city][:50] for row in g["prices"] for p in row if len(p)])
    med = float(np.median(allp))
    rng = np.random.default_rng(7)
    Us = []
    for g in D[city]:
        U, A, C = mc_utility(g, rng, med)
        Us.append(U)
    within = np.mean([U.std(axis=1).mean() for U in Us])
    best = None
    for lam in GRID:
        r = rev_c1_summary(Us, lam)
        cs = 100 * r["cost"] / r["cost_lm"]
        front.append(dict(city=city, lam=lam, pct=round(r["pct"], 2), cost_share=round(cs, 2)))
        if cs <= 100 / 3:
            best = (lam, r, cs)
    lam_m, rm, cs_m = best
    r5 = rev_c1_summary(Us, 5)
    sel = {}
    for rule in ["average", "least_misery", "most_pleasure"]:
        ps = [rev_pick(U, rule) for U in Us]
        sel[rule] = (np.mean([U[p].min() for U, p in zip(Us, ps)]), np.mean([U[p].mean() for U, p in zip(Us, ps)]))
    # mediation on the multi-criteria utility, with exact and noisy reactions
    med_rows = {}
    for kind, sd in [("exact", 0.0), ("noise sd=0.02", 0.02)]:
        rng2 = np.random.default_rng(11)
        Ts = [U + (rng2.normal(0, sd, U.shape) if sd else 0) for U in Us]
        bars = [T.min(1).max() - REV_MARGIN for T in Ts]
        lm = np.mean([T[rev_pick(U, "least_misery")].min() >= b for U, T, b in zip(Us, Ts, bars)])
        rng3 = np.random.default_rng(23)
        res = [rev_mediate(U, U=T, bar=b, rng=rng3, react_fn=rev_graded, parser=rev_parser_replay, memory=True)
               for U, T, b in zip(Us, Ts, bars)]
        med_rows[kind] = (100 * lm, 100 * np.mean([r["converged"] for r in res]),
                          np.mean([r["rounds"] for r in res if r["converged"]]))
    rows.append(dict(city=city, median_dish_price=round(med, 2), within_option_sd=round(within, 4),
                     achievable=round(rm["achievable"], 4), lam_matched=lam_m,
                     pct_matched=round(rm["pct"], 1), pct_lo=round(rm["pct_lo"], 1), pct_hi=round(rm["pct_hi"], 1),
                     cost_share_matched=round(cs_m, 1), gain_matched=round(rm["gain"], 4),
                     pct_lam5=round(r5["pct"], 1), cost_share_lam5=round(100 * r5["cost"] / r5["cost_lm"], 1),
                     avg_minsat=round(sel["average"][0], 4), lm_minsat=round(sel["least_misery"][0], 4),
                     mp_minsat=round(sel["most_pleasure"][0], 4),
                     lm_agree_exact=round(med_rows["exact"][0], 1), med_agree_exact=round(med_rows["exact"][1], 1),
                     med_rounds_exact=round(med_rows["exact"][2], 2),
                     lm_agree_noise=round(med_rows["noise sd=0.02"][0], 1),
                     med_agree_noise=round(med_rows["noise sd=0.02"][1], 1),
                     med_rounds_noise=round(med_rows["noise sd=0.02"][2], 2)))
    print(rows[-1])

for name, rr in [("e7_multicriteria.csv", rows), ("e7_frontier.csv", front)]:
    with open(f"{OUT}/{name}", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rr[0].keys())); w.writeheader(); w.writerows(rr)

{'city': 'Houston, TX', 'median_dish_price': 10.79, 'within_option_sd': np.float64(0.0872), 'achievable': np.float64(0.0162), 'lam_matched': np.float64(10.0), 'pct_matched': np.float64(64.3), 'pct_lo': 55.0, 'pct_hi': 72.7, 'cost_share_matched': np.float64(31.1), 'gain_matched': np.float64(0.0104), 'pct_lam5': np.float64(45.2), 'cost_share_lam5': np.float64(15.9), 'avg_minsat': np.float64(0.9177), 'lm_minsat': np.float64(0.9339), 'mp_minsat': np.float64(0.8068), 'lm_agree_exact': np.float64(100.0), 'med_agree_exact': np.float64(99.5), 'med_rounds_exact': np.float64(1.74), 'lm_agree_noise': np.float64(58.0), 'med_agree_noise': np.float64(96.5), 'med_rounds_noise': np.float64(2.47)}


{'city': 'Seattle, WA', 'median_dish_price': 12.99, 'within_option_sd': np.float64(0.0884), 'achievable': np.float64(0.0146), 'lam_matched': np.float64(8.254), 'pct_matched': np.float64(65.7), 'pct_lo': 54.8, 'pct_hi': 74.8, 'cost_share_matched': np.float64(27.2), 'gain_matched': np.float64(0.0096), 'pct_lam5': np.float64(42.7), 'cost_share_lam5': np.float64(12.3), 'avg_minsat': np.float64(0.9166), 'lm_minsat': np.float64(0.9312), 'mp_minsat': np.float64(0.7785), 'lm_agree_exact': np.float64(100.0), 'med_agree_exact': np.float64(100.0), 'med_rounds_exact': np.float64(1.81), 'lm_agree_noise': np.float64(67.5), 'med_agree_noise': np.float64(97.0), 'med_rounds_noise': np.float64(2.16)}


In [11]:
# LaTeX table rows (tab_utilities.tex) and the frontier CSVs, from the CSVs above,
# so that every number in the table traces to a CSV.
import pandas as pd
R = nbio.REV_OUT
T = os.path.join(nbio.REV_OUT, "tex")
os.makedirs(T, exist_ok=True)
H, S = "Houston, TX", "Seattle, WA"


def f(x, d=4):
    return f"{x:.{d}f}"


def lam(x):
    x = float(x)
    return f"{x:.2g}" if x < 10 else f"{x:.0f}"


u = pd.read_csv(f"{R}/e1_utilities.csv")
mc = pd.read_csv(f"{R}/e7_multicriteria.csv")
names = {"S": "Cosine, whole menu (paper)", "Dsafe": "Cosine, diner's safe dishes",
         "Se": "Euclidean similarity", "Sc": "Centred cosine",
         "D3": "Order-aware, top-3 safe dishes", "D": "Order-aware, best safe dish"}
lines = []
for key in ["S", "Dsafe", "Se", "Sc", "D3", "D"]:
    cells = [names[key]]
    for c in (H, S):
        r = u[(u.city == c) & (u.utility == key)].iloc[0]
        cells += [f(r.achievable), lam(r.lam_matched),
                  f"{r.pct_matched:.1f} [{r.pct_matched_lo:.1f}, {r.pct_matched_hi:.1f}]", f"{r.pct_lam5:.1f}"]
    lines.append(" & ".join(cells) + r" \\")
lines.append(r"\midrule")
cells = ["Multi-criteria (taste, price, choice)"]
for c in (H, S):
    r = mc[mc.city == c].iloc[0]
    cells += [f(r.achievable), lam(r.lam_matched), f"{r.pct_matched:.1f} [{r.pct_lo:.1f}, {r.pct_hi:.1f}]", f"{r.pct_lam5:.1f}"]
lines.append(" & ".join(cells) + r" \\")
open(f"{T}/tab_utilities.tex", "w").write("\n".join(lines) + "\n")

# frontier per-utility CSVs for pgfplots
fr = pd.read_csv(f"{R}/e1_frontier.csv"); f7 = pd.read_csv(f"{R}/e7_frontier.csv")
for c, tag in ((H, "hou"), (S, "sea")):
    for key in ["S", "Se", "Sc", "D3", "D"]:
        d = fr[(fr.city == c) & (fr.utility == key)].sort_values("lam")
        d = d[(d.cost_share.notna()) & (d.cost_share <= 100.5)]
        d[["cost_share", "pct"]].to_csv(f"{T}/front_{tag}_{key}.csv", index=False)
    d = f7[(f7.city == c)].sort_values("lam"); d = d[(d.cost_share.notna()) & (d.cost_share <= 100.5)]
    d[["cost_share", "pct"]].to_csv(f"{T}/front_{tag}_MC.csv", index=False)
print(open(f"{T}/tab_utilities.tex").read())

Cosine, whole menu (paper) & 0.0402 & 3.8 & 58.0 [51.3, 64.1] & 67.1 & 0.0358 & 5.6 & 64.7 [59.2, 69.7] & 61.9 \\
Cosine, diner's safe dishes & 0.0384 & 4.6 & 65.1 [59.0, 70.3] & 67.3 & 0.0344 & 5.6 & 65.7 [59.8, 71.0] & 63.4 \\
Euclidean similarity & 0.0166 & 12 & 46.8 [39.8, 53.6] & 18.9 & 0.0118 & 15 & 48.4 [41.3, 55.1] & 16.0 \\
Centred cosine & 0.3708 & 0.83 & 66.3 [62.3, 70.3] & 92.0 & 0.4028 & 0.83 & 68.7 [64.4, 72.9] & 93.5 \\
Order-aware, top-3 safe dishes & 0.0034 & 50 & 71.3 [59.5, 80.5] & 32.6 & 0.0018 & 68 & 57.3 [45.7, 67.1] & 7.0 \\
Order-aware, best safe dish & 0.0016 & 68 & 59.6 [47.8, 70.3] & 3.6 & 0.0017 & 83 & 73.0 [62.1, 81.7] & 3.3 \\
\midrule
Multi-criteria (taste, price, choice) & 0.0162 & 10 & 64.3 [55.0, 72.7] & 45.2 & 0.0146 & 8.3 & 65.7 [54.8, 74.8] & 42.7 \\



## Does taste similarity predict what Food.com users like or choose?

A check of the satisfaction function itself, on real ratings. For every Food.com
user with at least 10 star ratings, a seeded 20% of their rated recipes is held
out; the user's taste is built from the other 80% exactly as
`nbio.build_personas_foodcom` does (the mean taste of their 4-5 star recipes).
Two tests on the held-out recipes, each an AUC (0.5 = chance):

- **ratings**: does cosine(user, recipe) separate liked (4-5 stars) from not liked (1-3)?
- **choice**: does cosine rank a held-out recipe the user chose above a random recipe they never rated?

Each is reported for the personal taste vector, a non-personal baseline (the mean
taste of all users) and, for choice, a centred variant. Needs the Food.com
`RAW_interactions.csv` (see `data/README.md`). Writes
`figdata/ubereats-foodcom/satisfaction_check.csv`.

In [12]:
import csv, math, random

SAT_SEED = 7


def cos(a, b):
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    if na == 0 or nb == 0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (na * nb)


def mean_vec(vs):
    out = []
    for i in range(6):
        out.append(sum(v[i] for v in vs) / len(vs))
    return out


def satisfaction_check():
    ROOT = nbio.HERE
    rng = random.Random(SAT_SEED)
    by_user = {}
    with open(nbio.FOODCOM_INTERACTIONS, newline="") as f:
        for row in csv.DictReader(f):
            s = int(row["rating"])
            if s > 0:
                by_user.setdefault(int(row["user_id"]), []).append((int(row["recipe_id"]), s))
    taste = {}

    def T(r):
        if r not in taste:
            taste[r] = nbio.recipe_taste(r)
        return taste[r]

    known = lambda r: sum(T(r)) > 0
    all_ids = [r for r in nbio._recipes().keys()]
    split, pool = {}, []
    for u, v in by_user.items():
        if len(v) < nbio.FOODCOM_MIN_RATINGS:
            continue
        v = [x for x in v if known(x[0])]
        if len(v) < 5:
            continue
        rng.shuffle(v)
        k = max(1, len(v) // 5)
        test, train = v[:k], v[k:]
        liked = [T(r) for r, s in train if s >= nbio.FOODCOM_LIKE_STARS]
        if not liked:
            continue
        split[u] = (mean_vec(liked), test)
        pool += liked
    centre = mean_vec(pool)
    sub = lambda v: [x - c for x, c in zip(v, centre)]
    wins = {}
    counts = {}
    for name in ("rating_personal", "rating_everyone", "choice_personal", "choice_everyone", "choice_centred"):
        wins[name] = 0.0
        counts[name] = 0
    liked_c, notliked_c = {"p": [], "e": []}, {"p": [], "e": []}
    for u, (tu, test) in split.items():
        seen = {r for r, _ in by_user[u]}
        for r, s in test:
            cp, ce = cos(tu, T(r)), cos(centre, T(r))
            if s >= nbio.FOODCOM_LIKE_STARS:
                liked_c["p"].append(cp); liked_c["e"].append(ce)
            else:
                notliked_c["p"].append(cp); notliked_c["e"].append(ce)
            neg = rng.choice(all_ids)
            while neg in seen or not known(neg):
                neg = rng.choice(all_ids)
            for name, a, b in (("choice_personal", cos(tu, T(r)), cos(tu, T(neg))),
                               ("choice_everyone", cos(centre, T(r)), cos(centre, T(neg))),
                               ("choice_centred", cos(sub(tu), sub(T(r))), cos(sub(tu), sub(T(neg))))):
                if a > b:
                    wins[name] += 1.0
                elif a == b:
                    wins[name] += 0.5
                counts[name] += 1
    r2 = random.Random(SAT_SEED)
    for key, name in (("p", "rating_personal"), ("e", "rating_everyone")):
        n = 200000
        for _ in range(n):
            a, b = r2.choice(liked_c[key]), r2.choice(notliked_c[key])
            if a > b:
                wins[name] += 1.0
            elif a == b:
                wins[name] += 0.5
        counts[name] = n
    rows = []
    for name in wins:
        test, who = name.split("_")
        rows.append({"test": test, "taste_vector": who, "auc": round(wins[name] / counts[name], 4),
                     "n_pairs": counts[name], "n_users": len(split)})
    out = ROOT / "figdata" / "ubereats-foodcom" / "satisfaction_check.csv"
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    for r in rows:
        print(r)
    print(f"held-out ratings: liked {len(liked_c['p']):,}, not liked {len(notliked_c['p']):,} -> {out.relative_to(ROOT)}")

In [13]:
satisfaction_check()

{'test': 'rating', 'taste_vector': 'personal', 'auc': 0.4898, 'n_pairs': 200000, 'n_users': 12035}
{'test': 'rating', 'taste_vector': 'everyone', 'auc': 0.4892, 'n_pairs': 200000, 'n_users': 12035}
{'test': 'choice', 'taste_vector': 'personal', 'auc': 0.5382, 'n_pairs': 144253, 'n_users': 12035}
{'test': 'choice', 'taste_vector': 'everyone', 'auc': 0.5107, 'n_pairs': 144253, 'n_users': 12035}
{'test': 'choice', 'taste_vector': 'centred', 'auc': 0.5707, 'n_pairs': 144253, 'n_users': 12035}
held-out ratings: liked 136,745, not liked 7,508 -> figdata/ubereats-foodcom/satisfaction_check.csv
